In [1]:
%load_ext autoreload
%autoreload 2

from tasks.diffusion import GaussianDiffusionTask
import os
import torch
from tqdm import tqdm
from model.gaussian_diffusion import *
from evaluate import load

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["LATENT_CONTROL_CKPT_DIR"] = (
    "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints"
)

In [3]:
diffusion_task = GaussianDiffusionTask.load_from_checkpoint(
            os.path.join(
                "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints/kxm7kqwf/"
                "last.ckpt",
            ),
            strict=False,
        )

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.


In [4]:
diffusion_task.setup()

In [5]:
decoder = diffusion_task.decoder.to("cuda")
ema_model = diffusion_task.ema_model.module.to("cuda")

In [6]:
generations_str = []
for it in tqdm(range(1024//128)):
    z = sample(model=ema_model, batch_size=128, sampling_timesteps=100, sampler="ddpm", schedule=diffusion_task.sampling_schedule, diffusion_objective=diffusion_task.cfg.diffusion_objective)
    generations = decoder.generate(z=z.half(), max_length=150)
    generations_str += decoder.tokenizer.batch_decode(generations, skip_special_tokens=True)

100%|██████████| 8/8 [00:54<00:00,  6.82s/it]


In [7]:
mauve = load('mauve')

In [36]:
val_sentences = []
for batch in diffusion_task.val_dataloader():
    val_sentences += batch['input_str']

[autoreload of tasks.diffusion failed: Traceback (most recent call last):
  File "/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
  File "/cvmfs/ai.mila.quebec/apps/arch/distro/python/3.10/lib/python3.10/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 619, in _exec
  File "<frozen importlib._bootstrap_external>", line 879, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1017, in get_code
  File "<frozen importlib._bootstrap_external>", line 947, in source_to_code
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/home/mila/l/leo.gagnon/sentence_diffusion/task

In [40]:
feats

array([[ 0.28266448, -0.07071321, -0.12284269, ...,  0.46286353,
        -0.36809856, -1.5618408 ],
       [ 0.1997722 ,  0.03727617, -0.16265811, ...,  0.1928231 ,
        -0.07944007, -0.6413111 ],
       [ 0.12668873,  0.2290402 ,  0.3727111 , ...,  0.90355235,
        -0.12745844, -0.8761029 ],
       ...,
       [ 0.91976625,  0.32114068, -0.11162158, ...,  0.4863157 ,
         0.019405  , -1.2746114 ],
       [-0.09168848, -0.00516012,  0.498887  , ...,  0.7421001 ,
        -0.11295341, -0.39248955],
       [ 0.3999722 , -0.13580173,  0.17010427, ...,  0.7180514 ,
        -0.01828075, -0.8647044 ]], dtype=float32)

In [15]:
import mauve

In [22]:
from mauve.compute_mauve import get_features_from_input

In [ ]:
with torch.no_grad():
    feats = get_features_from_input(None, None, val_sentences, 'gpt2-large', 256, 0, 'q', 128)

ValueError: too many dimensions 'str'

In [42]:
with torch.no_grad():
    out = mauve.compute_mauve(p_text=generations_str, q_features=torch.Tensor(feats), device_id=0, max_text_length=256, batch_size=128)

Featurizing p:   0%|          | 0/8 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 154.00 MiB. GPU 0 has a total capacity of 44.64 GiB of which 104.44 MiB is free. Including non-PyTorch memory, this process has 44.53 GiB memory in use. Of the allocated memory 40.40 GiB is allocated by PyTorch, and 3.60 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [33]:
out.mauve

0.07757899835966037